# 03. 미분·compiler·hardware·기록·hybrid workflow

이 노트북은 서로 다른 전문 트랙을 하나의 검증 흐름으로 연결한다. 모든 수치와 모델은 교육용 toy example이며 실제 backend, compiler 또는 fault-tolerant resource estimator를 대체하지 않는다.

In [ ]:
import numpy as np
from collections import deque

np.set_printoptions(precision=6, suppress=True)
SEED = 20260913
rng = np.random.default_rng(SEED)

## 1. Parameter-shift gradient 감사

단일 $R_y(\theta)$ 뒤의 $Z$ expectation은 $f(\theta)=\cos(\theta)$다. 이 generator에서는 $[f(\theta+\pi/2)-f(\theta-\pi/2)]/2$가 정확한 미분이다. 모든 gate가 이 두 번 실행 공식에 해당하는 것은 아니다.

In [ ]:
def expectation_z(theta):
    return float(np.cos(theta))

def parameter_shift(theta):
    return 0.5 * (expectation_z(theta + np.pi / 2) - expectation_z(theta - np.pi / 2))

angles = np.linspace(-np.pi, np.pi, 17)
shift_gradients = np.array([parameter_shift(theta) for theta in angles])
analytic_gradients = -np.sin(angles)
assert np.allclose(shift_gradients, analytic_gradients, atol=1e-12)
print('analytic parameter-shift max error:', np.max(np.abs(shift_gradients - analytic_gradients)))

In [ ]:
def shot_expectation_z(theta, shots, random_generator):
    probability_zero = (1 + np.cos(theta)) / 2
    zero_count = random_generator.binomial(shots, probability_zero)
    return 2 * zero_count / shots - 1

def shot_parameter_shift(theta, shots, random_generator):
    plus = shot_expectation_z(theta + np.pi / 2, shots, random_generator)
    minus = shot_expectation_z(theta - np.pi / 2, shots, random_generator)
    return 0.5 * (plus - minus)

theta = 0.7
true_gradient = -np.sin(theta)
for shots in (100, 1_000, 10_000):
    estimates = np.array([shot_parameter_shift(theta, shots, rng) for _ in range(100)])
    print({
        'shots_per_shift': shots,
        'mean': float(estimates.mean()),
        'std': float(estimates.std(ddof=1)),
        'bias_vs_analytic': float(estimates.mean() - true_gradient),
        'circuit_evaluations': 200,
    })

assert abs(estimates.mean() - true_gradient) < 0.02

## 2. 의미를 보존하는 작은 compiler rewrite

인접한 자기 역원 gate 두 개만 취소한다. 최적화 후 gate 수가 줄었다는 사실만으로는 부족하며, 원본과 결과 unitary가 global phase까지 동등한지 검사한다.

In [ ]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
CNOT_01 = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
], dtype=complex)

def operation_matrix(operation):
    name, qubits = operation
    if name == 'cx' and qubits == (0, 1):
        return CNOT_01
    gate = {'x': X, 'z': Z, 'h': H}[name]
    return np.kron(gate, I) if qubits == (0,) else np.kron(I, gate)

def circuit_unitary(operations):
    result = np.eye(4, dtype=complex)
    for operation in operations:
        result = operation_matrix(operation) @ result
    return result

def cancel_adjacent_involutions(operations):
    output = []
    for operation in operations:
        if output and output[-1] == operation and operation[0] in {'x', 'z', 'h', 'cx'}:
            output.pop()
        else:
            output.append(operation)
    return output

def equivalent_up_to_global_phase(left, right, atol=1e-12):
    overlap = np.trace(left.conj().T @ right) / left.shape[0]
    if abs(overlap) < atol:
        return False
    phase = overlap / abs(overlap)
    return np.allclose(right, phase * left, atol=atol)

source_ir = [('h', (0,)), ('x', (1,)), ('x', (1,)), ('cx', (0, 1)), ('cx', (0, 1)), ('z', (1,))]
optimized_ir = cancel_adjacent_involutions(source_ir)
assert len(optimized_ir) < len(source_ir)
assert equivalent_up_to_global_phase(circuit_unitary(source_ir), circuit_unitary(optimized_ir))
print('before:', source_ir)
print('after: ', optimized_ir)

실제 pass는 parameter, classical control, measurement, timing과 target legality를 처리해야 한다. 최적화 모델이 후보를 선택하더라도 compiler verifier가 의미와 target 제약을 강제해야 한다.

## 3. Connectivity가 만드는 routing 비용

4-qubit 선형 topology에서 두 qubit 사이의 최단 거리를 구하고 단순 SWAP 상한을 계산한다. 이는 production router가 아니며 배치 변화와 되돌림 비용을 생략한 설명용 지표다.

In [ ]:
coupling_edges = {(0, 1), (1, 2), (2, 3)}

def shortest_distance(start, goal, edges):
    adjacency = {node: set() for edge in edges for node in edge}
    for left, right in edges:
        adjacency[left].add(right)
        adjacency[right].add(left)
    queue = deque([(start, 0)])
    visited = {start}
    while queue:
        node, distance = queue.popleft()
        if node == goal:
            return distance
        for neighbor in adjacency[node] - visited:
            visited.add(neighbor)
            queue.append((neighbor, distance + 1))
    raise ValueError('연결되지 않은 topology입니다.')

logical_two_qubit_gates = [(0, 3), (1, 2), (3, 0)]
routing_report = []
for control, target in logical_two_qubit_gates:
    distance = shortest_distance(control, target, coupling_edges)
    illustrative_swaps = max(0, distance - 1)
    routing_report.append({
        'logical_pair': (control, target),
        'distance': distance,
        'illustrative_swaps': illustrative_swaps,
        'illustrative_added_cx': 3 * illustrative_swaps,
    })

for row in routing_report:
    print(row)
assert routing_report[0]['distance'] == 3

## 4. 비동기 hybrid job과 idempotency

네트워크 QPU 호출은 동기 함수처럼 숨기지 않는다. 중복 제출, 재시도 가능한 실패와 checkpoint를 표현하는 최소 상태 저장소를 만든다.

In [ ]:
class HybridJobStore:
    def __init__(self):
        self.jobs = {}
        self.keys = {}

    def submit(self, idempotency_key, payload):
        if idempotency_key in self.keys:
            existing = self.jobs[self.keys[idempotency_key]]
            if existing['payload'] != payload:
                raise ValueError('같은 idempotency key에 다른 payload를 사용할 수 없습니다.')
            return existing
        job_id = f'job-{len(self.jobs) + 1}'
        job = {
            'id': job_id, 'key': idempotency_key, 'payload': payload,
            'state': 'QUEUED', 'attempt': 0, 'checkpoint': None,
        }
        self.jobs[job_id] = job
        self.keys[idempotency_key] = job_id
        return job

    def run(self, job_id, fail_first_attempt=False):
        job = self.jobs[job_id]
        if job['state'] == 'SUCCEEDED':
            return job
        job['state'] = 'RUNNING'
        job['attempt'] += 1
        job['checkpoint'] = {'completed_terms': 2 * job['attempt']}
        if fail_first_attempt and job['attempt'] == 1:
            job['state'] = 'RETRYABLE_FAILED'
        else:
            job['state'] = 'SUCCEEDED'
            job['result'] = {'expectation': -1.234}
        return job

store = HybridJobStore()
first = store.submit('experiment-A-iteration-7', {'shots': 2_000})
duplicate = store.submit('experiment-A-iteration-7', {'shots': 2_000})
assert first['id'] == duplicate['id']
try:
    store.submit('experiment-A-iteration-7', {'shots': 4_000})
except ValueError:
    pass
else:
    raise AssertionError('idempotency key 충돌을 거부해야 합니다.')
assert store.run(first['id'], fail_first_attempt=True)['state'] == 'RETRYABLE_FAILED'
final_job = store.run(first['id'], fail_first_attempt=True)
assert final_job['state'] == 'SUCCEEDED' and final_job['attempt'] == 2
print(final_job)

## 5. Fault-tolerant 자원 수치의 민감도

아래 식의 계수는 물리 모델이 아닌 **설명용 가상 계수**다. code distance와 factory 수를 바꾸면 qubit와 시간이 반대 방향으로 움직일 수 있음을 보여 줄 뿐이다. 논문·설계 결론에는 Microsoft Resource Estimator, Qualtran 같은 검증된 도구와 명시적인 hardware/QEC/error-budget 모델을 사용한다.

In [ ]:
def illustrative_ft_model(logical_qubits, t_count, code_distance, factories, cycle_microseconds=1.0):
    data_physical_qubits = logical_qubits * 2 * code_distance**2
    factory_physical_qubits = factories * 4 * code_distance**2
    seconds = t_count * 10 * code_distance * cycle_microseconds * 1e-6 / factories
    return {
        'distance': code_distance,
        'factories': factories,
        'illustrative_physical_qubits': data_physical_qubits + factory_physical_qubits,
        'illustrative_seconds': seconds,
    }

sensitivity = [
    illustrative_ft_model(50, 100_000_000, distance, factories)
    for distance in (15, 25, 35)
    for factories in (1, 4)
]
for row in sensitivity:
    print(row)

assert sensitivity[1]['illustrative_seconds'] < sensitivity[0]['illustrative_seconds']
assert sensitivity[-1]['illustrative_physical_qubits'] > sensitivity[0]['illustrative_physical_qubits']

## 확장 과제

1. 여러 generator spectrum에서 generalized parameter-shift가 필요한 사례를 조사한다.
2. rotation folding pass를 추가하고 random circuit differential test를 만든다.
3. 실제 target의 directed coupling, gate duration과 error rate로 cost function을 확장한다.
4. job state에 `CANCEL_REQUESTED`, `CANCELLED`, deadline과 retry budget을 추가한다.
5. OpenTelemetry trace에서 compile/job/backend snapshot ID를 연결한다.
6. 검증된 resource estimator에서 logical error budget과 factory 선택의 민감도를 비교한다.